# LLM agent – CA_1, FOODS_3

Goal: a planner asks questions in plain language; the model chooses which of my functions to call, and every number in the answer comes from those functions.

## Toy: one tool, one loop

In [11]:
import json
import sys
from pathlib import Path

import anthropic
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
load_dotenv(ROOT / ".env")

from src.forecasting.forecast import forecast

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-haiku-5-5"

In [12]:
def get_forecast_total(item: str, days: int = 7) -> dict:
    """Total median forecast (q50) for one item over the next `days` days."""
    try:
        fc = forecast(item, "CA_1", horizon=days)
    except (KeyError, ValueError) as e:
        return {"error": str(e)}
    return {
        "item": item,
        "from": str(fc["date"].iloc[0].date()),
        "to": str(fc["date"].iloc[-1].date()),
        "days": days,
        "q50_total_units": round(float(fc["q50"].sum()), 1),
    }


get_forecast_total("FOODS_3_090", 7)

{'item': 'FOODS_3_090',
 'from': '2016-05-23',
 'to': '2016-05-29',
 'days': 7,
 'q50_total_units': 368.6}

In [13]:
TOOL = {
    "name": "get_forecast_total",
    "description": (
        "Total forecast demand (median, q50) for one FOODS_3 item in store CA_1, "
        "over the next 1-28 days after the data ends (2016-05-22). "
        "Use it for any question about how much an item will sell."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "item": {"type": "string", "description": "Item id, e.g. FOODS_3_090"},
            "days": {"type": "integer", "minimum": 1, "maximum": 28},
        },
        "required": ["item"],
    },
}

In [14]:
def run_toy(question: str, verbose: bool = True) -> str:
    """Ask, run any tool the model asks for, send the result back, repeat until it answers."""
    messages = [{"role": "user", "content": question}]
    for step in range(5):  # safety limit
        response = client.messages.create(
            model=MODEL, max_tokens=1024, tools=[TOOL], messages=messages
        )
        # Keep the whole reply, thinking blocks included: the API needs them back
        messages.append({"role": "assistant", "content": response.content})
        if verbose:
            print(f"step {step}: stop_reason = {response.stop_reason}")

        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text")

        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = get_forecast_total(**block.input)
                if verbose:
                    print(f"  tool call: {block.name}({block.input}) -> {output}")
                results.append(
                    {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)}
                )
        messages.append({"role": "user", "content": results})
    return "Stopped: too many tool calls."

In [15]:
answer = run_toy("Quantas unidades de FOODS_3_090 vão vender na próxima semana?")
print("\nAnswer:", answer)

step 0: stop_reason = tool_use
  tool call: get_forecast_total({'item': 'FOODS_3_090', 'days': 7}) -> {'item': 'FOODS_3_090', 'from': '2016-05-23', 'to': '2016-05-29', 'days': 7, 'q50_total_units': 368.6}
step 1: stop_reason = end_turn

Answer: Para a próxima semana (de 23 a 29 de maio de 2016), a previsão mediana para o item FOODS_3_090 na loja CA_1 é de cerca de **369 unidades** (368,6).


In [16]:
print(run_toy("How much will FOODS_9_999 sell in the next 10 days?"))
print("\n" + "=" * 60 + "\n")
print(run_toy("Qual vai ser a procura de FOODS_3_090 nos próximos 40 dias?"))
print("\n" + "=" * 60 + "\n")
print(run_toy("Qual é o preço de compra do FOODS_3_090 ao fornecedor?"))

step 0: stop_reason = tool_use
  tool call: get_forecast_total({'item': 'FOODS_9_999', 'days': 10}) -> {'error': '"No forecast for item \'FOODS_9_999\' in store \'CA_1\'"'}
step 1: stop_reason = end_turn
I couldn't get a forecast for FOODS_9_999. The system has no forecast for that item in store CA_1, so I can't say how much it will sell in the next 10 days.

The item ID may be mistyped or may not be a FOODS_3 item. Forecasts are only available for FOODS_3 items, in the format FOODS_3_090. If you send me the correct ID, I'll get the 10-day forecast.


step 0: stop_reason = tool_use
  tool call: get_forecast_total({'item': 'FOODS_3_090', 'days': 28}) -> {'item': 'FOODS_3_090', 'from': '2016-05-23', 'to': '2016-06-19', 'days': 28, 'q50_total_units': 1580.5}
step 1: stop_reason = end_turn
A previsão mediana (q50) para FOODS_3_090 na loja CA_1 nos primeiros 28 dias, de 23 de maio a 19 de junho de 2016, é de **1 580,5 unidades**.

Não consigo obter a previsão para os 40 dias completos, porq

**What the toy shows**

- The model picks the tool and its inputs (it chose `days=7` for "next week"); my code runs it and returns the result.
- Errors returned as data let it recover: an unknown item is explained, 40 days is cut to the 28-day limit.
- With no tool for the question (purchase price), it declines instead of guessing.
- But numbers it works out itself can be wrong: for the 12 days after 2016-06-19 it wrote "20 to 31 June", a date that does not exist. Hence the rule for the real agent: every number must come from a tool, checked by code.

In [17]:
from src.agent.tools import run_tool

run_tool("get_inventory_policy", {"item": "FOODS_3_090", "policy": "(s, S)"})

{'item': 'FOODS_3_090',
 'policy': '(s, S)',
 'review_day': '2016-05-23',
 'demand_class': 'smooth',
 'price_usd': 1.6,
 'forecast_daily_units': 53.02,
 'rule': 'weekly review; order only when stock + on order <= s, then up to S',
 'z': 3.11,
 'reorder_point_s_units': 1263.3,
 'batch_eoq_units': 321.7,
 'order_up_to_S_units': 1585,
 'S_in_days_of_demand': 29.9,
 'S_stock_value_usd': 2536.0}

## The agent: six tools, one loop, a grounding check

`src/agent/agent.py` runs the same loop as the toy with six tools (`src/agent/tools.py`), a system prompt (`src/agent/prompt.py`) and a check that every number in the answer comes from a tool result (`src/agent/grounding.py`). Below, two traced conversations: a comparison with a follow-up in Portuguese, and a live run of the digital twin.

In [18]:
from src.agent.agent import ask


def show(question, history=None):
    """Ask the agent and print each tool call, the answer and the grounding check."""
    answer = ask(question, history)
    print(f"Q: {question}\n")
    for c in answer.tool_calls:
        print(f"  [{c.name}] {json.dumps(c.inputs)} ({c.seconds:.1f} s)")
    print(f"\nA: {answer.text}\n")
    print(
        f"  grounded: {not answer.ungrounded}, retried: {answer.retried}, "
        f"tokens: {answer.input_tokens} in / {answer.output_tokens} out"
    )
    return answer


first = show("Which policy loses the least sales in a 2-week supplier outage?")

Q: Which policy loses the least sales in a 2-week supplier outage?

  [compare_policies] {"scenario": "S1 supplier outage"} (0.0 s)

A: **Weeks of cover** loses the least sales in a 2-week supplier outage (S1). Its extra lost-sales cost versus normal is **970 USD** (±227 USD, 95% interval), with a fill rate of 0.952.

Ranking by extra lost-sales cost, from lowest to highest:

1. Weeks of cover: 970 USD (±227)
2. (R, S) normal: 1,154 USD (±484)
3. (s, S): 1,258 USD (±314)
4. (R, S) quantile: 1,438 USD (±406)

The intervals overlap, so the gap between the top three is not clearly significant.

Weeks of cover has the highest total cost including DC holding of the best performers (30,541 USD) and the lowest cycle service of the three best performers (0.900). (s, S) has the best fill rate (0.977) and the lowest total cost (23,905 USD), and the project's tests found it the most robust overall. Recovery takes about 44 days under (s, S) and about 36 days under Weeks of cover.

  grounded: True

In [19]:
_ = show("E qual é a taxa de serviço (fill rate) da (s, S) nesse cenário?", first.messages)

Q: E qual é a taxa de serviço (fill rate) da (s, S) nesse cenário?


A: A taxa de serviço (fill rate) da (s, S) no cenário S1 (falha de fornecedor de 14 dias) é de **0,977**.

Correção à minha resposta anterior: o custo total com a distribuição (DC) de 30.541 USD do Weeks of cover não é o mais alto dos quatro; o (R, S) quantile tem 32.041 USD e o (R, S) normal tem 44.582 USD.

  grounded: True, retried: False, tokens: 4500 in / 464 out


In [20]:
_ = show("Run a demand spike for FOODS_3_702 with weeks of cover.")

Q: Run a demand spike for FOODS_3_702 with weeks of cover.

  [run_scenario] {"item": "FOODS_3_702", "policy": "Weeks of cover", "scenario": "S2 demand spike"} (2.8 s)

A: Under the S2 demand spike (demand +50% for 10 days, starting 2016-03-27, seed 0), FOODS_3_702 with weeks of cover performed as follows, compared with its normal run over the 126 scored days (2016-01-18 to 2016-05-22):

| Metric | Spike | Normal | Change |
|---|---|---|---|
| Fill rate | 0.865 | 0.893 | −0.027 |
| Cycle service | 0.556 | 0.722 | −0.167 |
| Lost-sales cost | $154.35 | $119.25 | +$35.10 |
| Total cost incl. DC holding | $209.25 | $175.52 | +$33.73 |

The spike cost about $33.73 more in total, mostly from lost sales. This is a single item and a single seed, so it does not give a store-wide picture. For that, I can run compare_policies for the S2 demand spike scenario.

  grounded: True, retried: False, tokens: 5561 in / 654 out


## Evaluation: 30 questions

`src/agent/eval_set.json` holds 30 questions of 7 types, frozen before any tuning. The expected numbers are computed by the tools at run time. A question is correct only if the right tools were called, every expected number and name is in the answer, and the grounding check found no unsupported number. Run with `uv run python -m src.agent.run_eval`.

In [21]:
import pandas as pd

evaluation = pd.read_csv(ROOT / "results" / "agent_eval.csv")
checks = ["correct", "tools_ok", "numbers_ok", "names_ok", "grounded_ok"]
print(f"Score: {evaluation['correct'].sum()} / {len(evaluation)}")
print(f"Answers corrected by the grounding retry: {evaluation['retried'].sum()}")
print(
    f"Tokens: {evaluation['input_tokens'].sum():,} in / {evaluation['output_tokens'].sum():,} out"
)
evaluation.groupby("type")[checks].mean().round(2)

Score: 30 / 30
Answers corrected by the grounding retry: 2
Tokens: 189,772 in / 10,599 out


,correct,tools_ok,numbers_ok,names_ok,grounded_ok
type,,,,,
assumptions,1.0,1.0,1.0,1.0,1.0
comparison,1.0,1.0,1.0,1.0,1.0
follow-up,1.0,1.0,1.0,1.0,1.0
forecast,1.0,1.0,1.0,1.0,1.0
live scenario,1.0,1.0,1.0,1.0,1.0
out of scope,1.0,1.0,1.0,1.0,1.0
policy,1.0,1.0,1.0,1.0,1.0


## Findings

1. **30 / 30 with Claude Haiku 5.5**, every question type at 100%. The first run scored 29 / 30, and the one failure was the checker, not the agent: "1 134,8" (a Portuguese space as thousands separator) was read as two numbers.
2. **The code does the arithmetic; the model only chooses tools and phrases the answer.** The toy invented a date ("20 to 31 June"), and the first agent ranked costs wrongly. Both went away once the tools returned totals, changes against normal and ranks.
3. **The grounding retry earns its place:** it corrected 2 of the 30 answers before they were shown.
4. **Honest negative: the check verifies numbers, not claims.** In the app, the agent said weeks of cover had "the lowest fill rate of the four policies (0.9523)" in a supplier outage. The number is right, but it ranks third: (R, S) normal is lower at 0.8942. This happened despite the rank fields and the prompt rule, and the evaluation set, written by the author of the tools, did not catch it. Checking ranking words against the rank fields is the next step.
5. **A cheap model is enough when the tools do the work:** a full evaluation costs about 3 US cents, and at 30 / 30 a dearer model has nothing left to gain on this set.